# FujiCV Quickstart — Train CIFAR-10 in Minutes

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dsabarinathan/fujicv/blob/main/examples/quickstart.ipynb)

This notebook trains a **ResNet-18** on **CIFAR-10** using FujiCV.  
The dataset downloads automatically (~170 MB). No GPU required (but recommended).

**What you'll get in < 5 minutes:**
- ~80% val accuracy after 10 epochs  
- `best.pt` checkpoint  
- `history.csv` with per-epoch metrics  
- Grad-CAM visualization on test images

> Runtime → Change runtime type → **T4 GPU** for fastest results.

## 1. Install FujiCV

In [ ]:
# Install FujiCV (PyTorch is pre-installed in Colab)
!pip install fujicv --quiet

## 2. Imports

In [ ]:
import torch
from torch.utils.data import DataLoader

import fujicv
from fujicv.data.datasets import get_default_dataset
from fujicv.data.transforms import get_train_transforms, get_val_transforms
from fujicv.engine.trainer import Trainer
from fujicv.losses.classification import CrossEntropyLoss
from fujicv.metrics.classification import Accuracy
from fujicv.models.builder import ModelBuilder

print(f'FujiCV version : {fujicv.__version__}')
print(f'PyTorch version: {torch.__version__}')
print(f'Device         : {fujicv.get_device()}')

## 3. Load CIFAR-10 (auto-downloads)

In [ ]:
IMAGE_SIZE = 32    # CIFAR-10 native resolution
BATCH_SIZE = 128

train_ds, val_ds, class_to_idx = get_default_dataset(
    name='cifar10',
    root='/content/data',
    train_transform=get_train_transforms(IMAGE_SIZE, level='medium'),
    val_transform=get_val_transforms(IMAGE_SIZE),
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train : {len(train_ds):,} samples')
print(f'Val   : {len(val_ds):,} samples')
print(f'Classes: {list(class_to_idx.keys())}')

## 4. Build Model

In [ ]:
model = ModelBuilder(
    backbone_name='resnet18',
    backbone_source='timm',
    pretrained=True,
    task='classification',
    num_outputs=10,
    image_size=IMAGE_SIZE,
).build()

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'ResNet-18: {total_params:.1f} M parameters')

## 5. Train

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

EPOCHS = 10

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=CrossEntropyLoss(),
    metrics={'accuracy': Accuracy()},
    optimizer=optimizer,
    scheduler=scheduler,
    epochs=EPOCHS,
    task='classification',
    output_dir='/content/runs/cifar10',
    class_to_idx=class_to_idx,
    monitor_metric='val_accuracy',
    mixed_precision=True,
)

history = trainer.train()

## 6. Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history.metrics['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, history.metrics['train_loss'], label='Train loss')
ax1.plot(epochs, history.metrics['val_loss'],   label='Val loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Loss'); ax1.legend(); ax1.grid(True, alpha=0.4)

ax2.plot(epochs, history.metrics['train_accuracy'], label='Train acc')
ax2.plot(epochs, history.metrics['val_accuracy'],   label='Val acc')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy'); ax2.legend(); ax2.grid(True, alpha=0.4)

plt.suptitle('FujiCV — CIFAR-10 Training (ResNet-18)', fontsize=13)
plt.tight_layout()
plt.show()

best_acc = max(history.metrics['val_accuracy'])
print(f'\nBest val accuracy: {best_acc:.4f}  ({best_acc*100:.1f}%)')

## 7. Grad-CAM Visualization

In [ ]:
import numpy as np
from fujicv.eval.gradcam import GradCAM, overlay_heatmap

# Pick 4 val images
images, labels = [], []
for i in range(4):
    img, lbl = val_ds[i * 250]
    images.append(img)
    labels.append(lbl)

batch = torch.stack(images)   # (4, 3, 32, 32)

# Attach Grad-CAM to the last layer of the backbone
cam = GradCAM(model, target_layer=model.backbone.layer4[-1])

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
class_names = list(class_to_idx.keys())

for i in range(4):
    img_tensor = batch[i:i+1]   # (1, 3, 32, 32)
    heatmap    = cam.generate(img_tensor)

    # Original image (denormalize)
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    orig = (batch[i] * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

    axes[0, i].imshow(orig)
    axes[0, i].set_title(f'Label: {class_names[labels[i]]}')
    axes[0, i].axis('off')

    axes[1, i].imshow(orig)
    axes[1, i].imshow(heatmap, alpha=0.5, cmap='jet')
    axes[1, i].set_title('Grad-CAM')
    axes[1, i].axis('off')

plt.suptitle('Top row: original | Bottom row: Grad-CAM heatmap', fontsize=12)
plt.tight_layout()
plt.show()

## 8. Inference from Checkpoint

In [ ]:
import torch

# Load best checkpoint
ckpt = torch.load('/content/runs/cifar10/best.pt', map_location='cpu')
print('Checkpoint keys:', list(ckpt.keys()))
print('Best epoch     :', ckpt['epoch'])

# Quick batch prediction
model.eval()
model.load_state_dict(ckpt['model_state_dict'])

with torch.no_grad():
    logits = model(batch)
    preds  = logits.argmax(dim=-1)

for i in range(4):
    true  = class_names[labels[i]]
    pred  = class_names[preds[i].item()]
    conf  = torch.softmax(logits[i], dim=0).max().item()
    mark  = '✓' if true == pred else '✗'
    print(f'  {mark}  True: {true:<15}  Predicted: {pred:<15}  Confidence: {conf:.1%}')

## 9. What's Next?

| What | How |
|---|---|
| Better accuracy | `backbone_name='efficientnet_b0'` or `'convnext_tiny'` |
| ImageNet subset | `from fujicv.data.hf_dataset import load_hf_dataset` then `load_hf_dataset('Maysee/tiny-imagenet', ...)` |
| Hyperparameter search | `from fujicv.hpo import run_hpo` |
| LR Finder | `from fujicv.training import LRFinder` |
| ONNX export | `from fujicv.export import to_onnx, quantize_onnx` |
| Multi-GPU (DDP) | `torchrun --nproc_per_node=4 script.py` + `use_ddp=True` |

**GitHub**: https://github.com/dsabarinathan/fujicv  
**Docs**: https://dsabarinathan.github.io/fujicv  
**PyPI**: `pip install fujicv`